# 2주차 3교시 — 텍스트 임베딩과 벡터 유사도

「최신인공지능」 2026 · 실습 2·3·4

| 실습 | 내용 |
|------|------|
| 2 | 문장 임베딩과 코사인 유사도 |
| 3 ★ | 키워드 검색 vs 의미 검색 |
| 4 | 유사도 행렬 히트맵 |

**진행 방법**: 위에서부터 셀을 순서대로 실행합니다. 첫 셀에서 모델을 1회 다운로드합니다(약 1~2분).

## 0. 준비 — 패키지 설치와 모델 로딩

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 한국어 임베딩 모델 (768차원)
# 다국어 모델(paraphrase-multilingual-MiniLM-L12-v2)은 한국어 성능이 떨어져
# 실습 3에서 오답을 1위로 올린다. 이것이 곧 '한국어 지원'이 모델 선택 기준인 이유다.
model = SentenceTransformer("snunlp/KR-SBERT-V40K-klueNLI-augSTS")

def cosine(a, b):
    """두 벡터 사이 각도의 코사인. 1에 가까울수록 의미가 유사하다."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("준비 완료")

## 실습 2 — 임베딩과 코사인 유사도

> 질문과 정답 문서는 겹치는 단어가 거의 없다. 그런데도 유사도가 높게 나오는지 본다.

In [ ]:
sentences = [
    "졸업하려면 몇 학점 들어야 해?",        # 0
    "본 학과의 이수 요건은 총 130학점이다.",  # 1
    "오늘 학식 메뉴가 뭐야?",               # 2
]

vectors = model.encode(sentences)
print("벡터 shape:", vectors.shape)   # (3, 768) — 문장 3개, 각 768차원
print("0번 벡터 앞 5개:", vectors[0][:5])
print()
print("질문 vs 졸업요건 :", round(float(cosine(vectors[0], vectors[1])), 4))
print("질문 vs 학식메뉴 :", round(float(cosine(vectors[0], vectors[2])), 4))

### 관찰 포인트

| 확인할 것 | 의미 |
|----------|------|
| `shape` 가 `(3, 768)` | 문장 길이와 무관하게 **항상 768개 숫자**로 고정 |
| 졸업요건 유사도 > 학식 유사도 | **단어가 아니라 의미로 비교**되고 있다는 증거 |
| 벡터 값이 사람 눈에 무의미 | 사람이 해석하는 게 아니라 **비교용 좌표**일 뿐 |

> 정확한 수치는 모델·버전에 따라 달라집니다. **대소 관계**만 확인하면 됩니다.

## 실습 3 ★ — 키워드 검색 vs 의미 검색

> 이번 교시의 핵심 실습. RAG 가 왜 필요한지를 한 번에 보여준다.

In [ ]:
# 학과 안내 문서라고 가정한 5개 문장
docs = [
    "본 학과의 이수 요건은 총 130학점이다.",
    "전공필수 과목은 반드시 수강해야 하며 재수강이 제한된다.",
    "학생 식당은 평일 오전 11시부터 오후 2시까지 운영한다.",
    "도서관은 시험 기간에 24시간 개방한다.",
    "캡스톤디자인은 4학년 1학기에 이수하는 것을 권장한다.",
]

query = "졸업하려면 몇 학점 들어야 해?"

# ── 방법 A: 키워드 일치 검색 (Ctrl+F 방식) ──────────────
print("=" * 55)
print("[A] 키워드 검색")
print("=" * 55)
keywords = query.replace("?", "").split()   # ['졸업하려면', '몇', '학점', '들어야', '해']
for i, doc in enumerate(docs):
    hits = [kw for kw in keywords if kw in doc]
    print(f"  문서{i}: 일치 {len(hits)}개 {hits}")

# ── 방법 B: 의미 기반 검색 (임베딩) ─────────────────────
print()
print("=" * 55)
print("[B] 의미 검색")
print("=" * 55)

q_vec = model.encode(query)
d_vecs = model.encode(docs)

scores = [(i, float(cosine(q_vec, v))) for i, v in enumerate(d_vecs)]
scores.sort(key=lambda x: x[1], reverse=True)

for rank, (i, score) in enumerate(scores, 1):
    mark = " ★" if rank == 1 else ""
    print(f"  {rank}위  유사도 {score:.4f}  문서{i}: {docs[i]}{mark}")

### 관찰 포인트 ★

| 확인할 것 | 결론 |
|----------|------|
| 키워드 검색이 정답을 골라내는가? | **못 고릅니다.** 정답 문서0과 무관한 문서1이 **1개씩 동점** ― 순위를 매길 근거가 없음 |
| 문서1은 왜 걸렸는가? | "수강**해**야"에 "해"가 들어 있어서. 의미와 무관한 우연한 일치 |
| 의미 검색이 정답을 1위로 올렸는가? | **올립니다.** 단어가 하나도 안 겹쳐도 |
| 2위 문서는 왜 올라왔는가? | "수강·과목"이 졸업 요건과 의미적으로 가깝기 때문 |

> 이것이 **RAG 의 첫 단계**입니다. 10~11주차에서 수천 개 문서로 확대합니다.

### 확장 — 질문을 바꿔 보기

- "밥 언제 먹을 수 있어?" → 몇 번 문서가 1위가 될까?
- 키워드 검색이 **더 유리한** 경우는? (→ 정확한 고유명사·코드·모델명)
  이 지점이 11주차 **하이브리드 검색**의 동기가 됩니다.

In [ ]:
for q in ["밥 언제 먹을 수 있어?", "시험 기간에 공부할 데 없을까?"]:
    qv = model.encode(q)
    best = max(range(len(docs)), key=lambda i: cosine(qv, d_vecs[i]))
    print(f'"{q}"')
    print(f"   → 1위: 문서{best}  {docs[best]}")
    print()

## 실습 4 — 유사도 행렬 히트맵

> 한글이 깨지면 아래 폰트 설치 셀을 실행하고 **런타임을 재시작**합니다.
> 깨져도 실습 목적(블록 구조 확인)에는 지장이 없습니다.

In [ ]:
# (선택) 한글 폰트 — 실행 후 [런타임] > [세션 다시 시작]
# !apt-get -qq install fonts-nanum > /dev/null
# import matplotlib.font_manager as fm; fm._load_fontmanager(try_read_cache=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

texts = [
    "졸업 이수 학점",
    "학위 취득 요건",
    "전공필수 과목",
    "학생 식당 운영 시간",
    "점심 메뉴 안내",
]

vecs = model.encode(texts, normalize_embeddings=True)  # 정규화하면 내적 = 코사인
sim = vecs @ vecs.T                                     # 5x5 유사도 행렬

# 한글 폰트가 있으면 문장을, 없으면 번호를 라벨로 쓴다
installed = {f.name for f in fm.fontManager.ttflist}
korean = next((n for n in ["NanumGothic", "NanumBarunGothic", "Malgun Gothic"] if n in installed), None)
if korean:
    plt.rcParams["font.family"] = korean
    plt.rcParams["axes.unicode_minus"] = False
labels = [f"{i}: {t}" for i, t in enumerate(texts)] if korean else [f"text {i}" for i in range(len(texts))]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap="YlOrRd", vmin=0, vmax=1)

ax.set_xticks(range(len(texts)))
ax.set_yticks(range(len(texts)))
ax.set_xticklabels(range(len(texts)))
ax.set_yticklabels(labels)

for i in range(len(texts)):
    for j in range(len(texts)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=9)

plt.colorbar(im)
plt.title("Sentence Similarity Matrix")
plt.tight_layout()
plt.show()

### 관찰 포인트

```
       0     1     2     3     4
  0  1.00  0.8x  0.5x  0.1x  0.1x     ┐
  1  0.8x  1.00  0.5x  0.1x  0.1x     ├ 학사 관련 (0,1,2) 블록
  2  0.5x  0.5x  1.00  0.1x  0.2x     ┘
  3  0.1x  0.1x  0.1x  1.00  0.7x     ┐ 식당 관련 (3,4) 블록
  4  0.1x  0.1x  0.2x  0.7x  1.00     ┘
```

| 확인할 것 | 의미 |
|----------|------|
| 대각선이 전부 1.00 | 자기 자신과의 유사도 |
| **블록 구조가 보이는가** | 주제별로 자동으로 묶임 = 의미 공간이 잘 만들어짐 |
| 정규화 후 내적을 쓴 이유 | `normalize_embeddings=True` 면 코사인 유사도와 동일 |

## 핵심 정리

| 항목 | 요점 |
|------|------|
| 임베딩 | 텍스트 → **고정 길이 벡터**. 의미가 비슷하면 벡터도 가깝다 |
| 차원 | 모델마다 고정 (384·768·1024 등). 문장 길이와 무관 |
| 코사인 유사도 | 두 벡터의 **각도**로 측정. 1에 가까울수록 유사 |
| 왜 각도인가 | 문장 길이 영향을 없애고 **방향(의미)만** 비교 |
| 키워드 검색의 한계 | 단어가 안 겹치면 못 찾음 |
| 의미 검색 | 단어가 안 겹쳐도 찾아냄 → **RAG 의 1단계** |
| 주의 | 문서와 질문은 **같은 임베딩 모델**로 변환해야 함 |

> **3주차 예고**: 개발 환경을 정리하고 `ollama.chat()` 과 LangChain 의 `ChatOllama` 를
> 나란히 놓고 비교한 뒤, 첫 번째 체인을 만듭니다.